# Day-search results

Exploratory reporting on a `concopt search` run: the shape of the whole
candidate set (not just its top rows), and a close look at the winning
day's profile.

**Prerequisite** -- `concopt search` only writes its top rows by
default. Run it with `--out-all` first so the full ~31,000-candidate
frame is on disk for this notebook to load:

```
concopt search --pln tests/data/KJFKEGLL_CONC_01.pln \
               --npz data/era5/route_legs.npz \
               --surface-npz <your surface .npz, from era5.reduce_surface_to_npz> \
               --out results.csv --out-all results_all.csv
```

Everything computed here reuses concopt's own functions (`limits`,
`search.march_legs`, `report._step_climb_schedule`, ...) -- this
notebook loads and plots, it does not reimplement the march.

In [ ]:
import calendar

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from concopt import limits, report, runways
from concopt.atmos import isa, KT_TO_MS
from concopt.era5 import load_legs_npz
from concopt.route import build_legs, parse_pln, supersonic_segment
from concopt.search import (DECEL_DESCENT_S, DEPARTURE_TO_ACCEL_S, TARGET_FL,
                             _format_hmm, local_to_departure_utc, march_legs)

# Same colour for the same series in every cell below.
PALETTE = ["#4C72B0", "#DD8452", "#55A868"]  # blue, orange, green

PLN_PATH = "../tests/data/KJFKEGLL_CONC_01.pln"
NPZ_PATH = "../data/era5/route_legs.npz"
RESULTS_ALL_CSV = "../data/results_all.csv"  # from `concopt search --out-all`, see above
ACCEL_ID = "LINND"
DECEL_ID = "BARIX"
CRUISE_MACH = limits.CRUISE_MACH
TOP_N = 50  # how many rows count as "the shortlist" below

## 1. Load

In [ ]:
plan = parse_pln(PLN_PATH)
legs = build_legs(plan["waypoints"])
mask = supersonic_segment(legs, accel_id=ACCEL_ID, decel_id=DECEL_ID)
ss_idx = np.flatnonzero(mask)
ss_legs = [legs[i] for i in ss_idx]

df = pd.read_csv(RESULTS_ALL_CSV, parse_dates=["local_date", "departure_utc"])
# "" round-trips through CSV as NaN in an otherwise-string column.
for col in ("jfk_flag", "lhr_flag", "flags"):
    df[col] = df[col].fillna("")

print(f"{len(df):,} candidates, {df['local_date'].min().date()} to {df['local_date'].max().date()}")


def _still_air_reference(ss_legs, ss_idx, n_route_legs, cruise_mach):
    """Still-air, ISA+0 reference for the supersonic segment: the same
    march_legs the search itself uses, fed a synthetic zero-wind
    atmosphere. FL450-FL600 sits entirely inside the 11-20 km isothermal
    layer (atmos.isa), so one constant temperature reproduces ISA+0 at
    every one of the 4 raw ERA5 pressure levels without needing a
    pressure -> altitude inversion."""
    t_iso, _ = isa(TARGET_FL.mean() * 100.0 * 0.3048)
    level = np.array([150.0, 125.0, 100.0, 70.0])  # era5.py's mandatory levels
    time = np.array(["2015-01-01", "2030-01-01"], dtype="datetime64[ns]")
    shape = (len(time), len(level), n_route_legs)
    data = dict(time=time, level=level, u=np.zeros(shape), v=np.zeros(shape), t=np.full(shape, t_iso))
    dep_i8 = np.array([np.datetime64("2020-01-01T12:00:00", "ns").astype("int64")])
    legs_out, _weight_per_leg = march_legs(ss_legs, ss_idx, data, dep_i8, DEPARTURE_TO_ACCEL_S, cruise_mach)
    return float(legs_out["accumulated_s"][0]) - DEPARTURE_TO_ACCEL_S


still_air_s = _still_air_reference(ss_legs, ss_idx, len(legs), CRUISE_MACH)
print(f"still-air ISA+0 supersonic segment: {_format_hmm(still_air_s)} "
      f"({still_air_s / 60.0:.0f} min) -- for context against the wind-assisted times below")

## 2. Total block time, all candidates

In [ ]:
total_min = df["total_time_s"] / 60.0
top_n = df.nsmallest(TOP_N, "total_time_s")
top_min = top_n["total_time_s"] / 60.0
bins = np.histogram_bin_edges(total_min, bins=60)

best_min = total_min.min()
median_min = total_min.median()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(total_min, bins=bins, color=PALETTE[0], alpha=0.8, label=f"all candidates (n={len(df):,})")
ax.hist(top_min, bins=bins, color=PALETTE[1], alpha=0.9, label=f"top {TOP_N}")
ax.axvline(best_min, color=PALETTE[1], linestyle="--", linewidth=1)
ax.axvline(median_min, color=PALETTE[0], linestyle="--", linewidth=1)
ax.annotate(f"best {_format_hmm(best_min * 60.0)}", xy=(best_min, ax.get_ylim()[1]),
            xytext=(4, -12), textcoords="offset points", color=PALETTE[1])
ax.annotate(f"median {_format_hmm(median_min * 60.0)}", xy=(median_min, ax.get_ylim()[1]),
            xytext=(4, -28), textcoords="offset points", color=PALETTE[0])
ax.set_xlabel("total block time (min)")
ax.set_ylabel("candidates")
ax.set_title("Total block time, all candidates")
ax.legend()
fig.tight_layout()

## 3. Total time by month

In [ ]:
month = df["local_date"].dt.month
by_month = [df.loc[month == m, "total_time_s"].to_numpy() / 60.0 for m in range(1, 13)]

fig, ax = plt.subplots(figsize=(9, 4.5))
bp = ax.boxplot(by_month, tick_labels=[calendar.month_abbr[m] for m in range(1, 13)],
                 patch_artist=True, showfliers=False)
for patch in bp["boxes"]:
    patch.set_facecolor(PALETTE[0])
    patch.set_alpha(0.6)
ax.set_ylabel("total block time (min)")
ax.set_title("Total block time by month")
fig.tight_layout()

## 4. Total time vs along-track wind

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
sc = ax.scatter(df["mean_wind_kt"], df["total_time_s"] / 60.0, c=df["mean_isa_dev_k"],
                 cmap="Blues", s=10, alpha=0.6, edgecolors="none")
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("mean ISA deviation (K)")
ax.set_xlabel("mean along-track wind (kt)")
ax.set_ylabel("total block time (min)")
ax.set_title("Total time vs along-track wind, coloured by ISA deviation")
fig.tight_layout()

## 5. Top 10

In [ ]:
top10 = df.nsmallest(10, "total_time_s")
top10_display = pd.DataFrame({
    "date": top10["local_date"].dt.strftime("%Y-%m-%d"),
    "local_departure": top10["local_hour"].map(lambda h: f"{h:02d}:00"),
    "total_time": top10["total_time_s"].map(_format_hmm),
    "supersonic_time": top10["supersonic_time_s"].map(_format_hmm),
    "mean_fl": top10["mean_fl"].round(0).astype(int),
    "mean_wind_kt": top10["mean_wind_kt"].round(1),
    "runways": top10["jfk_runway"] + " / " + top10["lhr_runway"],
    "flags": top10["flags"].replace("", "-"),
})
top10_display.reset_index(drop=True)

## 6. Winning day: chosen FL vs ceiling

Reruns `march_legs` for the winning candidate only (the same call
`report.run_report` makes) to get per-sub-leg detail -- `results_all.csv`
only carries per-candidate means.

In [ ]:
winner = df.loc[df["total_time_s"].idxmin()]
winner_date = winner["local_date"].date()
winner_hour = int(winner["local_hour"])

npz_data = load_legs_npz(NPZ_PATH)
departure_utc = local_to_departure_utc(winner_date, winner_hour)
dep_i8 = np.array([pd.Timestamp(departure_utc).value], dtype="int64")

legs_out, weight_per_leg = march_legs(ss_legs, ss_idx, npz_data, dep_i8, DEPARTURE_TO_ACCEL_S, CRUISE_MACH)
leg = {k: v[0] for k, v in legs_out.items() if k not in ("accumulated_s", "weight_at_barix")}
weight_per_leg = weight_per_leg[0]  # weight at the start of each sub-leg

start_cum_nm = np.array([l.cum_nm - l.dist_nm for l in ss_legs])  # each sub-leg's own start
ceiling_fl = limits.ceiling_ft(weight_per_leg, leg["isa_dev_k"]) / 100.0
schedule = report._step_climb_schedule(ss_legs, leg["chosen_fl"])  # [(cum_nm, FL), ...] at each step

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.step(start_cum_nm, leg["chosen_fl"], where="post", color=PALETTE[0], label="chosen FL")
ax.plot(start_cum_nm, ceiling_fl, color=PALETTE[1], label="ceiling")
for step_cum_nm, step_fl in schedule:
    ax.annotate(f"FL{step_fl:.0f}", xy=(step_cum_nm, step_fl), xytext=(0, 6),
                textcoords="offset points", fontsize=8, ha="center")
ax.set_xlabel("distance along route (nm)")
ax.set_ylabel("flight level")
ax.set_title(f"Winning day {winner_date} {winner_hour:02d}:00 -- chosen FL vs ceiling")
ax.legend()
fig.tight_layout()

## 6a. Winning day: Mach limits

The chosen mach at each leg (blue) vs the limit imposed by cruise Mach, CAS
limit (530 kt above FL430), or max total temperature (127°C). The binding
limit (marked by color) shows which constraint most restricts the aircraft.

In [ ]:
from concopt.atmos import speed_of_sound

# Compute mach limit at each leg's chosen FL
# limits.max_mach already bundles cruise_mach / CAS / total_temp constraints
temp_k_at_best = leg["temp_c"] + 273.15
mach_limit = limits.max_mach(leg["chosen_fl"], temp_k_at_best, weight_per_leg, CRUISE_MACH)

# Map binding labels to colors
binding_colors = {
    "cruise_mach": PALETTE[1],
    "cas": PALETTE[2],
    "total_temp": "#8b4513",  # brown
    "ceiling": "#cccccc",
}
colors = np.array([binding_colors.get(b, "gray") for b in leg["binding"]])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(start_cum_nm, leg["mach"], color=PALETTE[0], marker=".", markersize=4, label="actual mach")
ax.plot(start_cum_nm, mach_limit, color="red", linestyle="--", linewidth=1.5, label="mach limit")
# Scatter actual mach colored by binding limit
ax.scatter(start_cum_nm, leg["mach"], c=colors, s=20, alpha=0.7, edgecolors="none")
ax.set_xlabel("distance along route (nm)")
ax.set_ylabel("Mach number")
ax.set_title(f"Winning day {winner_date} {winner_hour:02d}:00 -- Mach actual vs limit")
ax.set_ylim([1.8, 2.1])
# Custom legend for binding limits
from matplotlib.patches import Patch
legend_elements = [
    plt.Line2D([0], [0], color=PALETTE[0], marker=".", markersize=6, label="actual mach"),
    plt.Line2D([0], [0], color="red", linestyle="--", linewidth=1.5, label="limit (minimum of all)"),
    Patch(facecolor=PALETTE[1], label="binding: cruise_mach"),
    Patch(facecolor=PALETTE[2], label="binding: CAS (530 kt)"),
    Patch(facecolor="#8b4513", label="binding: total_temp (127°C)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8)
fig.tight_layout()

## 6b. Winning day: CAS limits

Converts actual mach at each leg to calibrated airspeed (CAS), and compares
against the 530 kt structural limit (the max CAS above FL430, where Concorde
must fly). Shows how much margin exists before hitting the CAS limit.

In [ ]:
def tas_to_cas(tas_ms, temp_k, p_pa):
    """Convert TAS (m/s) to CAS (m/s) given static temperature (K) and
    pressure (Pa). Uses the standard aerodynamic formula via mach."""
    from concopt.atmos import qc_over_p, A0, P0, GAMMA
    a_local = speed_of_sound(temp_k)
    mach = tas_ms / a_local
    qc = qc_over_p(mach) * p_pa
    # CAS is the tas_eq that produces the same impact pressure at sea level
    cas_ratio_sq = 2 / (GAMMA - 1) * (((qc / P0 + 1) ** ((GAMMA - 1) / GAMMA)) - 1)
    return A0 * np.sqrt(cas_ratio_sq)

# Pressure at each chosen FL
from concopt.atmos import isa as isa_func
_, pressure_pa = isa_func(leg["chosen_fl"] * 100.0 * 0.3048)
# TAS from stored tas_kt (already in knots, convert to m/s)
tas_ms = leg["tas_kt"] * KT_TO_MS
cas_ms = tas_to_cas(tas_ms, temp_k_at_best, pressure_pa)
cas_kt = cas_ms / KT_TO_MS

cas_limit_kt = 530.0

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(start_cum_nm, cas_kt, color=PALETTE[0], marker=".", markersize=4, label="actual CAS")
ax.axhline(cas_limit_kt, color="red", linestyle="--", linewidth=1.5, label=f"CAS limit ({cas_limit_kt:.0f} kt)")
ax.fill_between(start_cum_nm, 0, cas_kt, alpha=0.2, color=PALETTE[0])
ax.set_xlabel("distance along route (nm)")
ax.set_ylabel("CAS (kt)")
ax.set_title(f"Winning day {winner_date} {winner_hour:02d}:00 -- CAS actual vs limit")
ax.set_ylim([0, 600])
ax.legend()
fig.tight_layout()

## 6c. Winning day: Total temperature limits

Converts actual mach and static temperature at each leg to total (stagnation)
temperature, and compares against the 127°C structural limit. The temperature
margin is highest at lower altitudes where static temp is higher and mach is
lower.

In [ ]:
def total_temperature(mach, static_temp_k):
    """Stagnation (total) temperature given Mach and static temperature (K)."""
    return static_temp_k * (1.0 + 0.2 * mach ** 2)

total_temp_k = total_temperature(leg["mach"], temp_k_at_best)
total_temp_c = total_temp_k - 273.15
total_temp_limit_c = 127.0

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(start_cum_nm, total_temp_c, color=PALETTE[0], marker=".", markersize=4, label="actual total temp")
ax.axhline(total_temp_limit_c, color="red", linestyle="--", linewidth=1.5, label=f"limit ({total_temp_limit_c:.0f}°C)")
ax.fill_between(start_cum_nm, total_temp_c, total_temp_limit_c, alpha=0.2, color="red")
ax.set_xlabel("distance along route (nm)")
ax.set_ylabel("total temperature (°C)")
ax.set_title(f"Winning day {winner_date} {winner_hour:02d}:00 -- total temperature vs limit")
ax.set_ylim([-20, 140])
ax.legend()
fig.tight_layout()

## 7. Winning day: TAS, ground speed, along-track wind

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(start_cum_nm, leg["tas_kt"], color=PALETTE[0], label="TAS")
ax.plot(start_cum_nm, leg["gs_kt"], color=PALETTE[1], label="ground speed")
ax.plot(start_cum_nm, leg["wind_kt"], color=PALETTE[2], label="along-track wind")
ax.set_xlabel("distance along route (nm)")
ax.set_ylabel("speed (kt)")
ax.set_title(f"Winning day {winner_date} {winner_hour:02d}:00 -- TAS/GS/wind")
ax.legend()
fig.tight_layout()

## 8. Summary

In [ ]:
n = len(df)
jfk_unflyable = int((df["jfk_flag"] == "unflyable").sum())
lhr_unflyable = int((df["lhr_flag"] == "unflyable").sum())
jfk_flagged = int((df["jfk_flag"] == "xwind_25_30").sum())
lhr_flagged = int((df["lhr_flag"] == "xwind_25_30").sum())
lhr_westerly = int((df["lhr_runway"] == "27R/27L").sum())
lhr_easterly = int((df["lhr_runway"] == "09L/09R").sum())

print(f"Unflyable: JFK {jfk_unflyable:,} ({jfk_unflyable / n * 100:.1f}%), "
      f"LHR {lhr_unflyable:,} ({lhr_unflyable / n * 100:.1f}%) of {n:,} candidates")
print(f"Flagged {runways.XWIND_OK_KT:.0f}-{runways.XWIND_FLAG_KT:.0f} kt crosswind-gust band: "
      f"JFK {jfk_flagged:,} ({jfk_flagged / n * 100:.1f}%), "
      f"LHR {lhr_flagged:,} ({lhr_flagged / n * 100:.1f}%)")
print(f"LHR runway use: westerly (27R/27L) {lhr_westerly:,} ({lhr_westerly / n * 100:.0f}%), "
      f"easterly (09L/09R) {lhr_easterly:,} ({lhr_easterly / n * 100:.0f}%)")